# Week 7 - Notebook 2: Baseline Model Test

## Goal
Test the base Llama 3.2 model BEFORE fine-tuning:
1. Load base model with 4-bit quantization
2. Test on sample products
3. Evaluate on test set
4. Establish baseline performance

Expected result: ~$110 error (poor performance)

## Time: 15-20 minutes

In [ ]:
import sys
sys.path.append('..')

import os
import torch
from dotenv import load_dotenv
from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

from src.items import Item
from src.evaluator import evaluate
from src.config import config

# Load environment
load_dotenv()
hf_token = os.environ['HF_TOKEN']
login(hf_token, add_to_git_credential=True)

print("✅ Environment loaded")
print(f"GPU available: {torch.cuda.is_available()}")

## Configuration

In [ ]:
config.display()

## Load Test Data

In [ ]:
print(f"Loading test data from: {config.DATASET_NAME}")
_, _, test = Item.from_hub(config.DATASET_NAME)

print(f"✅ Loaded {len(test):,} test items")

## Load Base Model with 4-bit Quantization

In [ ]:
# Configure 4-bit quantization (QLoRA)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

print(f"Loading base model: {config.BASE_MODEL}")
print("This may take a few minutes...")

tokenizer = AutoTokenizer.from_pretrained(config.BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(
    config.BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

# Set padding token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
    model.config.pad_token_id = model.config.eos_token_id

print("✅ Model loaded")
print(f"Memory footprint: {model.get_memory_footprint() / 1e9:.2f} GB")

## Test on Sample Products

In [ ]:
def predict_base(item: Item) -> str:
    """
    Predict price using base model (no fine-tuning)
    """
    # Create prompt
    if not item.prompt:
        item.make_prompts(tokenizer, config.MAX_TOKENS, do_round=False)
    
    prompt = item.test_prompt()
    
    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=10,
            temperature=0.1,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
    
    # Decode
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    
    # Extract completion
    completion = response.split(config.PREFIX)[-1].strip()
    
    return completion

In [ ]:
# Test on a few examples
print("Testing base model on 5 sample products:\n")

for i in range(5):
    item = test[i]
    prediction = predict_base(item)
    
    print(f"Product: {item.title[:50]}...")
    print(f"Actual: ${item.price:.2f}")
    print(f"Predicted: {prediction}")
    print("-" * 60)

## Full Evaluation on Test Set

In [ ]:
# Evaluate on full test set
print(f"Evaluating on {config.EVAL_SIZE} test items...")
print("This will take 10-15 minutes...\n")

results = evaluate(
    predict_base,
    test,
    size=config.EVAL_SIZE,
    workers=1  # Sequential for GPU
)

print(f"\n{'='*60}")
print("BASE MODEL RESULTS (BEFORE FINE-TUNING)")
print(f"{'='*60}")
print(f"Average Error: ${results['average_error']:.2f}")
print(f"MSE: {results['mse']:,.0f}")
print(f"R²: {results['r2']:.1f}%")
print(f"{'='*60}")

## Visualize Results

In [ ]:
import plotly.graph_objects as go

# Compare with other models (from Week 6)
baseline_results = [
    ("Constant", "gray", 106.18),
    ("Linear Regression", "gray", 101.56),
    ("NLP + LR", "gray", 76.81),
    ("Random Forest", "gray", 72.28),
    ("XGBoost", "gray", 68.23),
    ("Human (Ed)", "black", 87.62),
    ("Neural Network", "orange", 63.97),
    ("GPT 4.1 Nano", "slateblue", 62.51),
    ("Grok 4.1 Fast", "slateblue", 57.62),
    ("Gemini 3 Pro", "slateblue", 50.54),
    ("Claude 4.5 Sonnet", "slateblue", 47.10),
    ("GPT 5.1", "slateblue", 44.74),
    ("Deep Neural Network", "orange", 46.49),
    ("Base Llama 3.2 4-bit", "darkred", results['average_error']),
]

labels, colors, values = zip(*baseline_results)

fig = go.Figure(go.Bar(x=labels, y=values, marker_color=colors))

fig.update_layout(
    title="Base Llama 3.2 vs Other Models - Price Prediction Error",
    yaxis=dict(range=[0, max(values)], title="Mean Absolute Error ($)"),
    xaxis=dict(tickangle=-45),
    width=1200,
    height=600,
)

fig.show()

print(f"\n⚠️ Base Llama performs poorly: ${results['average_error']:.2f} error")
print(f"   Worse than simple constant baseline!")
print(f"\n💡 This is expected - the model hasn't been trained for this task")
print(f"   Fine-tuning should reduce error to ~$40-65")

## Summary

✅ Baseline established!

**Base Model Performance:**
- Error: ~$110 (very poor)
- Worse than simple baselines
- Model doesn't understand the task

**Why so bad?**
- Llama 3.2 wasn't trained for price prediction
- No domain knowledge about products
- Needs fine-tuning to learn the task

**What fine-tuning will do:**
- Teach model to predict prices from product info
- Learn patterns in training data
- Expected improvement: 60-70% error reduction
- Target: $40-65 error (competitive with GPT-5.1)

**Next step:** `03_qlora_training.ipynb` - Fine-tune with QLoRA on Google Colab